# รวม `processed/prepared/`

`column_prepare.ipynb` คลีนเสร็จแล้วเขียนออก `processed/prepared/*.csv` — แต่ยังเป็น **7 ตารางแยก คนละ granularity**
(1 อุปกรณ์มีหลายชั้น หลายขั้นตอน หลาย property)

สมุดนี้ยุบทั้งหมดเป็น **1 แถว = 1 อุปกรณ์ (44 แถว)** แล้วสำรวจว่าข้อมูลที่ได้หน้าตาเป็นยังไง ใช้อะไรได้บ้าง

## Setup

In [21]:
import re
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)

ROOT = Path.cwd() if (Path.cwd() / "processed" / "prepared").exists() else Path.cwd().parent
SRC = ROOT / "processed" / "prepared"
tp = {p.stem: pd.read_csv(p) for p in sorted(SRC.glob("*.csv"))}

lay, mat, pr = tp["layers"], tp["materials"], tp["process"]
prop, stab, ven = tp["properties"], tp["stability"], tp["vendors"]

print("โหลดจาก:", SRC)
display(pd.DataFrame({n: {"rows": d.shape[0], "cols": d.shape[1]} for n, d in tp.items()}).T)

โหลดจาก: d:\University\Year 4 Semester 1 (Intern)\Perovskites-Research\processed\prepared


,rows,cols
devices,44,4
layers,352,10
materials,543,13
process,702,9
properties,563,16
stability,62,20
vendors,644,8


## 1. ตารางต้นทางเป็นระดับไหน

ก่อนรวมต้องรู้ว่าแต่ละตาราง 1 แถว = อะไร ถ้ายุบผิดระดับจะได้เลขผิดแบบเงียบ ๆ

In [22]:
for name, d in tp.items():
    n_dev = d["device_id"].nunique() if "device_id" in d.columns else 0
    per = round(len(d) / n_dev, 1) if n_dev else 0
    print(f"{name:12s} {len(d):4d} แถว | {n_dev:2d} devices | เฉลี่ย {per:5.1f} แถว/device")

devices        44 แถว | 44 devices | เฉลี่ย   1.0 แถว/device
layers        352 แถว | 44 devices | เฉลี่ย   8.0 แถว/device
materials     543 แถว | 44 devices | เฉลี่ย  12.3 แถว/device
process       702 แถว | 44 devices | เฉลี่ย  16.0 แถว/device
properties    563 แถว | 44 devices | เฉลี่ย  12.8 แถว/device
stability      62 แถว | 35 devices | เฉลี่ย   1.8 แถว/device
vendors       644 แถว | 44 devices | เฉลี่ย  14.6 แถว/device


## 2. `layers` → โครงสร้างสแต็ก

ดึงวัสดุของแต่ละชั้นหลัก + ธงว่ามีชั้น treatment ไหม

In [23]:
_L = lay.dropna(subset=["material"])

stack = _L.groupby("device_id").agg(n_layers=("layer_id", "nunique"))
for lt, tag in [("HTL", "htl"), ("ETL", "etl"), ("PSK", "psk"),
                ("Anode", "anode"), ("Cathode", "cathode")]:
    g = _L[_L["layer_type"] == lt].groupby("device_id")
    stack = stack.join(g["material"].first().rename(f"{tag}_material"))
    stack = stack.join(g["material_family"].first().rename(f"{tag}_family"))

stack = stack.join(_L[_L["layer_type"] == "PSK"]
                   .groupby("device_id")["thickness_nm_parsed"].first().rename("psk_thickness_nm"))

# ชั้น treatment: สนใจแค่ "มี/ไม่มี" -- ชื่อสารแต่ละตัวอยู่ใน paper เดียว ใช้เป็น category ไม่ได้
for lt, tag in [("HTL treatment", "has_htl_treatment"),
                ("PSK treatment", "has_psk_treatment"),
                ("ETL treatment", "has_etl_treatment")]:
    stack[tag] = stack.index.isin(_L.loc[_L["layer_type"] == lt, "device_id"])

print(f"stack: {stack.shape[0]} devices x {stack.shape[1]} คอลัมน์")
display(stack.head(10))

stack: 44 devices x 15 คอลัมน์


,n_layers,htl_material,htl_family,etl_material,etl_family,psk_material,psk_family,anode_material,anode_family,cathode_material,cathode_family,psk_thickness_nm,has_htl_treatment,has_psk_treatment,has_etl_treatment
device_id,,,,,,,,,,,,,,,
10.1002admi.202500231_Sample1,6,NiOx NPs,NiOx,C60,C60,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],NaN,ITO,TCO,Ag,Metal electrode,425.0,False,False,True
10.1002admi.202500231_Sample2,7,NiOx NPs,NiOx,C60,C60,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],NaN,ITO,TCO,Ag,Metal electrode,425.0,True,False,True
10.1002admi.202500231_Sample3,7,NiOx NPs,NiOx,C60,C60,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],NaN,ITO,TCO,Ag,Metal electrode,425.0,True,False,True
10.1002admi.202500231_Sample4,7,NiOx NPs,NiOx,C60,C60,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],NaN,ITO,TCO,Ag,Metal electrode,425.0,True,False,True
10.1016j.jechem.2021.11.029_Sample1,7,NiOx,NiOx,PCBM,PCBM,(FA0.85MA0.15)0.95Cs0.05Pb(I0.85Br0.15)3,NaN,ITO,TCO,Ag,Metal electrode,NaN,False,True,True
10.1016j.jechem.2021.11.029_Sample2,8,NiOx,NiOx,PCBM,PCBM,(FA0.85MA0.15)0.95Cs0.05Pb(I0.85Br0.15)3,NaN,ITO,TCO,Ag,Metal electrode,NaN,True,True,True
10.1016j.nanoen.2021.106285_Sample1,8,NiOx,NiOx,PCBM,PCBM,(FA0.83MA0.17)0.95Pb(I0.9Br0.1)3,NaN,FTO,TCO,Ag,Metal electrode,NaN,True,True,True
10.1016j.nanoen.2021.106285_Sample2,8,NiOx,NiOx,PCBM,PCBM,Cs0.04FA0.83MA0.12Pb(I0.88Br0.12)3,NaN,FTO,TCO,Ag,Metal electrode,NaN,True,True,True
10.1016j.nanoen.2021.106285_Sample3,8,NiOx,NiOx,PCBM,PCBM,KI-Cs0.04FA0.83MA0.12Pb(I0.88Br0.12)3,NaN,FTO,TCO,Ag,Metal electrode,NaN,True,True,True


In [24]:
stack.columns

Index(['n_layers', 'htl_material', 'htl_family', 'etl_material', 'etl_family', 'psk_material', 'psk_family', 'anode_material', 'anode_family', 'cathode_material', 'cathode_family', 'psk_thickness_nm',
       'has_htl_treatment', 'has_psk_treatment', 'has_etl_treatment'],
      dtype='str')

## 3. `materials` → ตัวทำละลายของชั้น PSK

`solvent_ratio` มาจาก section K ของ `column_prepare` (normalize แล้ว `8:2` กับ `800/200 µL` เป็น `4.0` เหมือนกัน)
เอาเฉพาะของชั้น PSK เพราะเป็น process knob ที่ส่งผลต่อ grain size

In [25]:
_psk_lid = lay.loc[lay["layer_type"] == "PSK", ["device_id", "layer_id"]]
_pm = mat.merge(_psk_lid, on=["device_id", "layer_id"])          # เฉพาะส่วนผสมของชั้น PSK

solv = (_pm.dropna(subset=["solvent_system"]).groupby("device_id")
          .agg(psk_solvent=("solvent_system", lambda s: s.mode().iat[0]),
               psk_solvent_ratio=("solvent_ratio", "first")))

print(f"ได้ตัวทำละลายของชั้น PSK: {len(solv)}/44 devices")
display(solv.groupby(["psk_solvent", "psk_solvent_ratio"], dropna=False).size().rename("n_devices").to_frame())

ได้ตัวทำละลายของชั้น PSK: 40/44 devices


,,n_devices
psk_solvent,psk_solvent_ratio,
DMF:DMSO (3.08:1),3.08,3
DMF:DMSO (4:1),4.00,37


## 4. `process` → นับขั้นตอน

⚠️ **แยกขั้นตอนเตรียมวัตถุดิบออก** — 4 devices จาก paper `10.1002admi.202500231` มี sheet พิเศษ `NiOx synthesis`
(สังเคราะห์อนุภาค NiOx เอง: Dissolve → pH adjust → Wash → Dry → Anneal) ซึ่ง**ไม่ใช่ขั้นตอนสร้างอุปกรณ์**
ถ้านับรวมจะทำให้ 4 ตัวนั้นดูมีขั้นตอนเยอะกว่าชาวบ้าน 3-5 ก้าวโดยไม่มีเหตุผลจริง

แยกไปเป็น `n_material_synth_steps` ต่างหาก (ไม่ทิ้ง แต่ไม่ปน)

In [26]:
_dev_steps = pr[pr["source_sheet"] == "Process"]        # ตัด NiOx synthesis ออก

proc = _dev_steps.groupby("device_id").agg(n_process_steps=("step_order", "count"))
for name, key in [("n_spin_coating", "spin coating"), ("n_annealing", "annealing")]:
    proc[name] = _dev_steps[_dev_steps["process_type"].str.lower() == key].groupby("device_id").size()

# เทียบแบบ lower เสมอ -- canonical ของ section C คือ "AntiSolvent" (S ใหญ่) เทียบตรง ๆ จะ miss
proc["has_antisolvent"] = (_dev_steps[_dev_steps["process_type"].str.lower() == "antisolvent"]
                           .groupby("device_id").size().gt(0))
proc["n_material_synth_steps"] = pr[pr["source_sheet"] != "Process"].groupby("device_id").size()

print("process_type ที่พบ (หลัง canonical):")
display(_dev_steps["process_type"].value_counts().head(10).rename("n_rows").to_frame())
display(proc.head(3))

process_type ที่พบ (หลัง canonical):


,n_rows
process_type,
Spin coating,213
Annealing,157
Thermal evaporation,82
Wash,48
AntiSolvent,42
Dry,28
Plasma Treat,21
UV/Ozone Treat,14
Stir,8


,n_process_steps,n_spin_coating,n_annealing,has_antisolvent,n_material_synth_steps
device_id,,,,,
10.1002admi.202500231_Sample1,13,3,2,True,5.0
10.1002admi.202500231_Sample2,14,3,2,True,5.0
10.1002admi.202500231_Sample3,14,3,2,True,5.0


## 5. `properties` → pivot long เป็น wide

`properties` เป็น EAV (1 แถว = 1 ค่า) ยุบด้วย `base_property` ที่แตกไว้ใน §3.1

**ใช้ `max`** = เอาค่าที่ดีที่สุด (champion) ต่ออุปกรณ์ — ถ้าจะใช้ค่าเฉลี่ยต้องกรอง `stat == "average"` ก่อน

In [27]:
KEEP = ["PCE", "Voc", "Jsc", "FF", "Grain size", "Trap density", "Rs", "Rrec", "HI"]

_p = prop.dropna(subset=["value_parsed"]).query("base_property in @KEEP")
jv = _p.pivot_table(index="device_id", columns="base_property", values="value_parsed", aggfunc="max")
jv.columns = [c.lower().replace(" ", "_") for c in jv.columns]

# เทียบ "มีแถว" กับ "แปลงเป็นตัวเลขได้" -- ต่างกันคือมีค่าว่างในต้นฉบับ
_cov = pd.DataFrame({
    "n_rows": prop.query("base_property in @KEEP").groupby("base_property").size(),
    "devices_มีแถว": prop.query("base_property in @KEEP").groupby("base_property")["device_id"].nunique(),
    "devices_ใช้ได้": _p.groupby("base_property")["device_id"].nunique(),
})
_cov["หาย"] = _cov["devices_มีแถว"] - _cov["devices_ใช้ได้"]
display(_cov.sort_values("devices_ใช้ได้", ascending=False))

,n_rows,devices_มีแถว,devices_ใช้ได้,หาย
base_property,,,,
FF,67,43,43,0
PCE,74,43,43,0
Jsc,67,43,43,0
Voc,67,43,43,0
Trap density,26,23,23,0
Rs,22,22,22,0
Rrec,15,15,15,0
Grain size,26,23,13,10
HI,11,11,11,0


## 6. `stability`

In [28]:
st = (stab.groupby("device_id")
        .agg(retention_pct=("retention_pct_parsed", "max"),
             stab_time_hr=("time_hr", "max"),
             stab_temp_c=("temperature_c_parsed", "max"),
             stab_humidity_pct=("humidity_pct_parsed", "max")))
print(f"มีข้อมูลความเสถียร {len(st)}/44 devices")
display(st.describe().T.round(1))

มีข้อมูลความเสถียร 35/44 devices


,count,mean,std,min,25%,50%,75%,max
retention_pct,28.0,79.4,16.5,40.0,69.4,85.5,90.1,100.8
stab_time_hr,33.0,1386.8,1073.3,72.0,1000.0,1200.0,1500.0,5000.0
stab_temp_c,28.0,55.7,28.0,25.0,25.0,65.0,85.0,85.0
stab_humidity_pct,20.0,46.8,9.2,35.0,40.0,40.0,55.0,60.0


## 7. รวมทั้งหมด

In [29]:
dev = tp["devices"][["device_id", "doi_raw", "sample_no"]].copy()
dev["paper"] = dev["device_id"].str.rsplit("_Sample", n=1).str[0]

ml = dev.set_index("device_id").join([stack, solv, proc, jv, st]).reset_index()

for c in ["n_spin_coating", "n_annealing", "n_material_synth_steps"]:
    ml[c] = ml[c].fillna(0).astype(int)
ml["has_antisolvent"] = ml["has_antisolvent"].fillna(False).astype(bool)

print(f"ml: {ml.shape[0]} แถว x {ml.shape[1]} คอลัมน์  |  {ml['paper'].nunique()} papers")
display(ml.head())

ml: 44 แถว x 39 คอลัมน์  |  15 papers


,device_id,doi_raw,sample_no,paper,n_layers,htl_material,htl_family,etl_material,etl_family,psk_material,psk_family,anode_material,anode_family,cathode_material,cathode_family,psk_thickness_nm,has_htl_treatment,has_psk_treatment,has_etl_treatment,psk_solvent,psk_solvent_ratio,n_process_steps,n_spin_coating,n_annealing,has_antisolvent,n_material_synth_steps,ff,grain_size,hi,jsc,pce,rrec,rs,trap_density,voc,retention_pct,stab_time_hr,stab_temp_c,stab_humidity_pct
0,10.1002admi.202500231_Sample1,10.1002admi.202500231,1,10.1002admi.202500231,6,NiOx NPs,NiOx,C60,C60,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],NaN,ITO,TCO,Ag,Metal electrode,425.0,False,False,True,DMF:DMSO (4:1),4.0,13,3,2,True,5,76.0,NaN,NaN,20.97,16.31,NaN,0.68,NaN,1.020,88.90,72.0,NaN,NaN
1,10.1002admi.202500231_Sample2,10.1002admi.202500231,2,10.1002admi.202500231,7,NiOx NPs,NiOx,C60,C60,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],NaN,ITO,TCO,Ag,Metal electrode,425.0,True,False,True,DMF:DMSO (4:1),4.0,14,3,2,True,5,75.0,NaN,NaN,22.01,17.11,NaN,0.64,NaN,1.040,61.37,72.0,NaN,NaN
2,10.1002admi.202500231_Sample3,10.1002admi.202500231,3,10.1002admi.202500231,7,NiOx NPs,NiOx,C60,C60,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],NaN,ITO,TCO,Ag,Metal electrode,425.0,True,False,True,DMF:DMSO (4:1),4.0,14,3,2,True,5,78.0,NaN,NaN,21.86,16.97,NaN,0.62,NaN,1.000,69.53,72.0,NaN,NaN
3,10.1002admi.202500231_Sample4,10.1002admi.202500231,4,10.1002admi.202500231,7,NiOx NPs,NiOx,C60,C60,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],NaN,ITO,TCO,Ag,Metal electrode,425.0,True,False,True,DMF:DMSO (4:1),4.0,14,3,2,True,5,79.0,NaN,NaN,20.79,16.25,NaN,0.53,NaN,0.990,82.46,72.0,NaN,NaN
4,10.1016j.jechem.2021.11.029_Sample1,10.1016j.jechem.2021.11.029,1,10.1016j.jechem.2021.11.029,7,NiOx,NiOx,PCBM,PCBM,(FA0.85MA0.15)0.95Cs0.05Pb(I0.85Br0.15)3,NaN,ITO,TCO,Ag,Metal electrode,NaN,False,True,True,DMF:DMSO (4:1),4.0,13,6,2,True,0,79.6,283.0,NaN,21.32,17.83,NaN,NaN,6.080000e+15,1.065,86.00,2880.0,NaN,NaN


## 8. ข้อมูลที่ได้หน้าตาเป็นยังไง

### 8.1 ความครบถ้วน

In [30]:
miss = (ml.isna().mean().mul(100).round(1).rename("missing_%").to_frame()
          .assign(**{"มีค่า": (ml.notna().sum())})
          .sort_values("missing_%"))
display(miss)

,missing_%,มีค่า
device_id,0.0,44
doi_raw,0.0,44
sample_no,0.0,44
paper,0.0,44
n_layers,0.0,44
htl_material,0.0,44
htl_family,0.0,44
etl_material,0.0,44
etl_family,0.0,44
psk_material,0.0,44


### 8.2 คอลัมน์ตัวเลข

In [31]:
_num = ml.select_dtypes("number").drop(columns=["sample_no"], errors="ignore")
display(_num.describe().T.round(2))

,count,mean,std,min,25%,50%,75%,max
n_layers,44.0,7.110000e+00,7.500000e-01,6.000000e+00,7.000000e+00,7.000000e+00,8.000000e+00,8.000000e+00
psk_thickness_nm,11.0,4.890900e+02,1.123100e+02,4.250000e+02,4.250000e+02,4.500000e+02,4.500000e+02,7.150000e+02
psk_solvent_ratio,40.0,3.930000e+00,2.500000e-01,3.080000e+00,4.000000e+00,4.000000e+00,4.000000e+00,4.000000e+00
n_process_steps,44.0,1.489000e+01,2.030000e+00,1.100000e+01,1.400000e+01,1.450000e+01,1.600000e+01,2.000000e+01
n_spin_coating,44.0,4.840000e+00,1.310000e+00,3.000000e+00,4.000000e+00,5.000000e+00,6.000000e+00,7.000000e+00
n_annealing,44.0,3.570000e+00,1.250000e+00,2.000000e+00,3.000000e+00,3.000000e+00,4.250000e+00,6.000000e+00
n_material_synth_steps,44.0,4.500000e-01,1.450000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,5.000000e+00
ff,43.0,7.772000e+01,4.700000e+00,6.539000e+01,7.510000e+01,7.846000e+01,8.119000e+01,8.468000e+01
grain_size,13.0,5.382000e+02,2.026700e+02,2.830000e+02,4.300000e+02,5.116100e+02,5.800000e+02,1.000000e+03
hi,11.0,4.460000e+00,3.180000e+00,7.500000e-01,2.680000e+00,3.300000e+00,6.200000e+00,1.090000e+01


### 8.3 คอลัมน์ที่ใช้เป็น feature ไม่ได้

2 กรณีที่ต้องคัดออกก่อนเทรน
1. **variance = 0** — ทุกอุปกรณ์ค่าเดียวกัน อธิบายอะไรไม่ได้
2. **ผูกกับ paper เดียว** — confound กับ DOI พอทำ Leave-One-Publication-Out จะใช้ไม่ได้

In [32]:
rows = []
for c in ml.columns:
    if c in ("device_id", "doi_raw", "paper", "sample_no"):
        continue
    s = ml[c].dropna()
    if s.empty:
        rows.append({"column": c, "n_unique": 0, "n_papers": 0, "สถานะ": "ว่างทั้งคอลัมน์"})
        continue
    nu = s.nunique()
    npap = ml.loc[s.index, "paper"].nunique()
    if nu <= 1:
        stt = "variance = 0 (ตัดทิ้ง)"
    elif npap <= 1:
        stt = "อยู่ paper เดียว (confound กับ DOI)"
    elif ml[c].notna().sum() < 15:
        stt = "ข้อมูลน้อยเกิน (<15 devices)"
    else:
        stt = "ใช้ได้"
    rows.append({"column": c, "n_unique": nu, "n_papers": npap,
                 "n_devices": int(ml[c].notna().sum()), "สถานะ": stt})

_chk = pd.DataFrame(rows).sort_values(["สถานะ", "column"]).reset_index(drop=True)
display(_chk["สถานะ"].value_counts().rename("n_columns").to_frame())
display(_chk)

,n_columns
สถานะ,
ใช้ได้,28
variance = 0 (ตัดทิ้ง),3
ข้อมูลน้อยเกิน (<15 devices),3
ว่างทั้งคอลัมน์,1


,column,n_unique,n_papers,n_devices,สถานะ
0,anode_family,1,15,44.0,variance = 0 (ตัดทิ้ง)
1,cathode_family,1,15,44.0,variance = 0 (ตัดทิ้ง)
2,htl_family,1,15,44.0,variance = 0 (ตัดทิ้ง)
3,grain_size,13,5,13.0,ข้อมูลน้อยเกิน (<15 devices)
4,hi,11,5,11.0,ข้อมูลน้อยเกิน (<15 devices)
5,psk_thickness_nm,3,3,11.0,ข้อมูลน้อยเกิน (<15 devices)
6,psk_family,0,0,NaN,ว่างทั้งคอลัมน์
7,anode_material,2,15,44.0,ใช้ได้
8,cathode_material,3,15,44.0,ใช้ได้
9,etl_family,2,15,44.0,ใช้ได้


### 8.4 target กับ mediator ของงานนี้

ตาม research goal: หา process ที่ขับ **grain size** (mediator) → **PCE / ความเสถียร** (target)

In [33]:
for col, lab in [("pce", "PCE (target หลัก)"),
                 ("grain_size", "Grain size (mediator)"),
                 ("retention_pct", "Retention (ความเสถียร)")]:
    s = ml[col].dropna()
    print(f"{lab:28s} {len(s):2d}/44 devices | {ml.loc[s.index, 'paper'].nunique():2d} papers | "
          f"{s.min():.2f} - {s.max():.2f}")

print()
print("มีครบทั้ง process feature + mediator + target พร้อมกันกี่ตัว:")
_core = ml.dropna(subset=["pce", "grain_size"])
print(f"  pce + grain_size           : {len(_core):2d} devices | {_core['paper'].nunique()} papers")
_core2 = ml.dropna(subset=["pce", "grain_size", "psk_solvent_ratio"])
print(f"  + psk_solvent_ratio        : {len(_core2):2d} devices | {_core2['paper'].nunique()} papers")

PCE (target หลัก)            43/44 devices | 15 papers | 13.98 - 26.50
Grain size (mediator)        13/44 devices |  5 papers | 283.00 - 1000.00
Retention (ความเสถียร)       28/44 devices | 13 papers | 40.00 - 100.85

มีครบทั้ง process feature + mediator + target พร้อมกันกี่ตัว:
  pce + grain_size           : 13 devices | 5 papers
  + psk_solvent_ratio        :  9 devices | 4 papers


## 9. บันทึกไว้ใช้ต่อ (ถ้าต้องการ)

In [34]:
# OUT = ROOT / "processed" / "device_level.csv"
# # ml.to_csv(OUT, index=False)    # <- เอา comment ออกเมื่อพร้อมเขียนจริง
# print("ยังไม่ได้เขียน — ปลด comment บรรทัดบนถ้าต้องการบันทึกไป", OUT)